In [1]:
import pandas as pd

In [2]:
covid_df = pd.read_csv('data/covid_df.csv', sep=';')
display(covid_df.head(2))

,date,country,confirmed,deaths,recovered,active,daily_confirmed,daily_deaths,daily_recovered,total_vaccinations,people_vaccinated,people_vaccinated_per_hundred,people_fully_vaccinated,people_fully_vaccinated_per_hundred,daily_vaccinations,vaccines,death_rate,recover_rate
0,2020-02-24,Afghanistan,1.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,2020-02-25,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0


Документация
* https://plotly.com/python/plotly-express/
* https://github.com/santosjorge/cufflinks

In [3]:
import plotly
import plotly.express as px
print(plotly.__version__)

6.5.2


Работа с plotly.express напоминает работу с библиотекой Seaborn. Отличие лишь в том, что все настройки графика (размеры, подписи осей, текста на графике) прописываются в самом методе.

### С помощью экспресс-режима (px) можно строить уже знакомые нам графики:

* line() — линейные графики;
https://plotly.com/python-api-reference/generated/plotly.express.line.html
* histogram() — гистограммы;
https://plotly.com/python-api-reference/generated/plotly.express.histogram.html
* scatter() — диаграммы рассеяния;
https://plotly.com/python-api-reference/generated/plotly.express.scatter
* box() — коробчатые диаграммы;
https://plotly.github.io/plotly.py-docs/generated/plotly.express.box.html
* bar() — столбчатые диаграммы;
https://plotly.com/python-api-reference/generated/plotly.express.bar.html
* pie() — круговые диаграммы.
https://plotly.com/python/pie-charts/

Также есть множество других графиков — их список вы можете посмотреть в документации.
https://plotly.com/python/

Примечание. Дальнейшая работа будет вестись с таблицей covid_df — полными данными о статистике распространения вируса Covid-19, а также о вакцинации в разных странах.

Рассмотрим процесс визуализации на примере. Посмотрим, как выглядит линейный график, построенный с помощью метода line() из модуля express. В документации к методу приведена пара десятков его параметров (они схожи с параметрами других методов) — мы приведём основные из них.

### Основные параметры метода line()

* data_frame — DataFrame, по которому строится график;
* x — признак по оси абсцисс;
* y — признак по оси ординат;
* height — высота графика;
* width — ширина графика;
* title — название графика.

Построим график роста зафиксированного числа случаев заражения (confirmed), смертей (deaths), выздоровлений (recovered) и активных случаев (active) за всё время. Для этого просуммируем статистику по дням и передадим полученный DataFrame в метод line().

Для отображения созданной методом line() фигуры используется метод **fig.show():**

In [ ]:
line_data = covid_df.groupby('date', as_index=False).sum()
fig = px.line(
    data_frame=line_data, #DataFrame
    x='date', #ось абсцисс
    y=['confirmed', 'recovered', 'deaths', 'active'], #ось ординат
    height=500, #высота
    width=900, #ширина
    title='Confirmed, Recovered, Deaths, Active cases over Time' #заголовок
)
fig.show()

---------

Давайте рассмотрим ещё один пример — построим столбчатую диаграмму, показывающую ТОП-10 стран по среднему проценту выздоравливающих пациентов (recover_rate). Для этого используем метод ```bar()``` модуля express. Добавим несколько параметров:

* ```color``` — группирующий признак, в соответствии с которым будут раскрашены столбцы диаграммы;
* ```text``` — текст, который будет подписан на столбцах диаграммы;
* ```orientation``` — ориентация графика ('v' — вертикальная, 'h' — горизонтальная).

In [5]:
#считаем средний процент выздоровлений для каждой страны
bar_data = covid_df.groupby(
    by='country',
    as_index=False
)[['recover_rate']].mean().round(2).nlargest(10, columns=['recover_rate'])

#строим график
fig = px.bar(
    data_frame=bar_data, #датафрейм
    x="country", #ось x
    y="recover_rate", #ось y
    color='country', #расцветка в зависимости от страны
    text = 'recover_rate', #текст на столбцах
    orientation='v', #ориентация графика
    height=500, #высота
    width=1000, #ширина
    title='Top 10 Countries for Recovery Rate' #заголовок
)

#отображаем его
fig.show()

-------

А теперь давайте построим что-нибудь, специфичное для библиотеки Plotly. Например, график ```treemap()``` (древесная, или иерархическая, диаграмма). Такой график используется для исследования показателя, когда число возможных категорий велико (например, число стран в таблице covid_df).

### Основные параметры метода treemap()

* data_frame — DataFrame, по которому строится график;
* path — категориальные признаки (их может быть несколько), в разрезе которых строится диаграмма;
* values — показатель, по которому рассчитываются размеры прямоугольников.

Построим иерархическую диаграмму для среднего ежедневного показателя выздоровевших пациентов (daily_recovered) во всех странах.

In [6]:
#считаем среднее ежедневно фиксируемое количество выздоровевших по странам
treemap_data = covid_df.groupby(
    by='country',
    as_index=False
)[['daily_recovered']].mean()

#строим график
fig = px.treemap(
    data_frame=treemap_data, #DataFrame
    path=['country'], #категориальный признак, для которого строится график
    values='daily_recovered', #параметр, который сравнивается
    height=500, #высота
    width=1000, #ширина
    title='Daily Recovered Cases by Country' #заголовок
)

#отображаем график
fig.show()

---

## Анимация графиков во времени

С помощью ```plotly.express``` можно строить даже анимированные графики. Мы рассмотрим только базовые приёмы анимации, но на самом деле это очень интересная и глубокая тема. Если вы захотите, то сможете ознакомиться с ней более детально здесь и здесь.
 *    https://plotly.com/python/animations/
 *   https://www.geeksforgeeks.org/data-visualization/animated-data-visualization-using-plotly-express/

Для нашей задачи отлично подойдёт график под названием ```choropleth()``` (тепловая картограмма) — это тепловая карта, которая строится на географической карте мира. Чтобы увидеть, как изменяется значение показателя на карте во времени, можно добавить в график анимацию.
* https://plotly.com/python-api-reference/generated/plotly.express.choropleth.html

### Основные параметры метода choropleth()

* data_frame — DataFrame, по которому строится график;
* locations — название столбца, из которого берутся локации (столбец со странами или регионами);
* locationmode — режим геопривязки; определяет, как будет производиться сопоставление данных с картой в Plotly (возможно сопоставление по названию страны, "country_name", или по её трёхзначному шифру, согласно международному стандарту ISO-3);
* range_color — диапазон изменения цвета;
* animation_frame — анимирующий признак, изменяя который, мы получаем визуализацию во времени;
* color_continuous_scale — цветовая палитра.

Итак, построим фоновую картограмму, которая покажет распространение (confirmed) коронавируса в мире во времени.

Предварительно для правильного отображения на анимационном бегунке даты в таблице covid_df необходимо перевести обратно в строковый тип данных.

In [7]:
#преобразуем даты в строковый формат
choropleth_data = covid_df.sort_values(by='date')
choropleth_data['date'] = choropleth_data['date'].astype('string')

#строим график
fig = px.choropleth(
    data_frame=choropleth_data, #DataFrame
    locations="country", #столбец с локациями
    locationmode = "country names", #режим сопоставления локаций с базой Plotly
    color="confirmed", #от чего зависит цвет
    animation_frame="date", #анимационный бегунок
    range_color=[0, 30e6], #диапазон цвета
    title='Global Spread of COVID-19', #заголовок
    width=800, #ширина
    height=500, #высота
    color_continuous_scale='Reds' #палитра цветов
)

#отображаем график
fig.show()
fig.write_html('data/plotly/choropleth.html')

C:\Users\1\AppData\Local\Temp\ipykernel_1980\3969230569.py:6: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



--------

## Трёхмерная визуализация

На самом деле общий принцип построения 3D-графиков ничем не отличается от построения обычных. Просто добавляется ещё один параметр — ось z (ось аппликат).

Построим 3D-диаграмму рассеяния, которая покажет, как число ежедневно обнаруживаемых случаев и число ежедневных смертей влияют на желание людей вакцинироваться. Для того чтобы нам было проще рассматривать диаграмму (точки будут более сгруппированными), построим её в логарифмическом масштабе по осям абсцисс и ординат.

Для построения такой диаграммы используем метод ```scatter_3d()```. Добавим несколько параметров:
* https://plotly.com/python-api-reference/generated/plotly.express.scatter_3d
* z — параметр по оси аппликат;
* log_x — установка логарифмического масштаба по оси x;
* log_y — установка логарифмического масштаба по оси y.

In [8]:
#фильтруем таблицу по странам
countries=['United States', 'Russia', 'United Kingdom', 'Brazil', 'France']
scatter_data = covid_df[covid_df['country'].isin(countries)]

#строим график
fig = px.scatter_3d(
    data_frame=scatter_data, #DataFrame
    x = 'daily_confirmed', #ось абсцисс
    y = 'daily_deaths', #ось ординат
    z = 'daily_vaccinations', #ось аппликат
    color='country', #расцветка в зависимости от страны
    log_x=True, 
    log_y=True,
    width=1000,
    height=700
)

#отображаем график
fig.show()
fig.write_html('data/plotly/scatter_3d.html')

## Сохранение графика plotly

Чтобы сохранить интерактивный график, построенный в библиотеке Plotly, чаще всего используется метод фигуры fig.write_html('path/to/file.html'), который сохраняет график в формате HTML, после чего вы можете вставлять его на свой сайт, в веб-приложение или просто делиться им с коллегами. 

Сохраним график трёхмерной диаграммы рассеяния:

In [9]:
fig.write_html('data/plotly/scatter_3d.html')

-------------

Принципы построения других базовых типов графиков 
https://plotly.com/python/basic-charts/

Все способы визуализации статистических показателей и зависимостей
https://plotly.com/python/statistical-charts/

----------

Задание 7.5

Постройте линейный график, который отображает, как изменялось ежедневное количество вакцинированных (daily_vaccinations) в мире во времени. Из графика найдите, чему равно количество вакцинированных (в миллионах) 28 февраля 2021 года (2021-02-28). Ответ округлите до целого числа.

In [10]:
line_data = covid_df.groupby('date', as_index=False).sum()
fig = px.line(
    data_frame=line_data, #DataFrame
    x='date', #ось абсцисс
    y=['daily_vaccinations'], #ось ординат
    height=500, #высота
    width=900, #ширина
    #title='Confirmed, Recovered, Deaths, Active cases over Time' #заголовок
)
fig.show()

Задание 7.6

Постройте анимированную тепловую картограмму для числа поставленных вакцин во всём мире (total_vaccinations). На полученной карте найдите, чему равно количество вакцинированных в Японии (Japan) на 24 марта 2021 года (2021-03-24). Ответ приведите в тысячах (без нулей) и округлите до целого числа.
Примечание. Если в jupyter notebook в VS Code не запускается анимация тепловой карты, попробуйте отобразить график командой fig.show(renderer='notebook').

In [11]:
#преобразуем даты в строковый формат
choropleth_data = covid_df.sort_values(by='date')
choropleth_data['date'] = choropleth_data['date'].astype('string')

#строим график
fig = px.choropleth(
    data_frame=choropleth_data, #DataFrame
    locations="country", #столбец с локациями
    locationmode = "country names", #режим сопоставления локаций с базой Plotly
    color="total_vaccinations", #от чего зависит цвет
    animation_frame="date", #анимационный бегунок
    range_color=[0, 30e6], #диапазон цвета
    title='total_vaccinations of COVID-19', #заголовок
    width=800, #ширина
    height=500, #высота
    color_continuous_scale='Reds' #палитра цветов
)

#отображаем график
fig.show()

C:\Users\1\AppData\Local\Temp\ipykernel_1980\2896390160.py:6: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



----------------

# plotly cufflinks

In [13]:
# Standard plotly imports
import plotly as py
import plotly.graph_objs as go
from plotly.offline import iplot, init_notebook_mode
# Using plotly + cufflinks in offline mode
import cufflinks
cufflinks.go_offline(connected=True)
init_notebook_mode(connected=True)